# 02 多项式插值

本节讨论第二章的主要全局多项式工具：

* 拉格朗日基函数；
* 牛顿差商；
* 等距节点上的 Runge 现象；
* 切比雪夫节点带来的改进。


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["font.sans-serif"] = [
    "Arial Unicode MS",
    "PingFang SC",
    "Heiti SC",
    "SimHei",
    "Noto Sans CJK SC",
    "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False

ROOT = pathlib.Path.cwd()
while not (ROOT / "src" / "py_sc").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from py_sc import (
    chebyshev_nodes,
    divided_difference_table,
    lagrange_interpolate,
    newton_divided_differences,
    newton_interpolate,
)


## 拉格朗日基函数

拉格朗日插值（Lagrange interpolation）写作

$$
p_n(x)=\sum_{j=0}^{n}y_jL_j(x),\qquad
L_j(x)=\prod_{m\ne j}\frac{x-x_m}{x_j-x_m}.
$$

每个基函数满足 $L_j(x_i)=\delta_{ij}$，也就是说它在自己的节点处取 1，在其他节点处取 0。


In [ ]:
def lagrange_basis(nodes, basis_index, x_eval):
    values = np.ones_like(x_eval, dtype=float)
    xj = nodes[basis_index]
    for m, xm in enumerate(nodes):
        if m != basis_index:
            values *= (x_eval - xm) / (xj - xm)
    return values

nodes = np.array([-1.0, -0.3, 0.4, 1.0])
x_plot = np.linspace(-1.1, 1.1, 500)

plt.figure(figsize=(8, 4.5))
for j in range(nodes.size):
    plt.plot(x_plot, lagrange_basis(nodes, j, x_plot), label=f"L_{j}")
plt.scatter(nodes, np.zeros_like(nodes), color="black", zorder=3)
plt.xlabel("x")
plt.ylabel("基函数值")
plt.title("拉格朗日基函数")
plt.legend()
plt.tight_layout()
plt.show()


## 牛顿差商

牛顿插值（Newton interpolation）写作

$$
p_n(x)=c_0+c_1(x-x_0)+c_2(x-x_0)(x-x_1)+\cdots.
$$

系数来自牛顿差商表的第一行。这个形式在逐步加入新节点时很方便，因为已有的低阶系数可以继续使用。


In [ ]:
x = np.array([0.0, 1.0, 2.0, 3.0])
y = np.array([1.0, 2.0, 0.0, 4.0])
x_eval = np.linspace(0.0, 3.0, 300)

lagrange_values = lagrange_interpolate(x, y, x_eval)
newton_nodes, coefficients = newton_divided_differences(x, y)
newton_values = newton_interpolate(newton_nodes, coefficients, x_eval)
_, table = divided_difference_table(x, y)

print("牛顿插值系数:", coefficients)
print("差商表第一行:", table[0])

plt.figure(figsize=(8, 4.5))
plt.plot(x_eval, lagrange_values, label="拉格朗日形式")
plt.plot(x_eval, newton_values, "--", label="牛顿形式")
plt.scatter(x, y, color="black", zorder=3, label="数据点")
plt.xlabel("x")
plt.ylabel("p(x)")
plt.title("拉格朗日形式与牛顿形式表示同一个多项式")
plt.legend()
plt.tight_layout()
plt.show()


## Runge 现象

插值误差公式中含有节点乘积

$$
\prod_{i=0}^{n}(x-x_i).
$$

在等距节点上做高次插值时，这一项可能在区间端点附近变大，从而导致明显振荡。Runge 函数是检验这一问题的经典例子。


In [ ]:
def runge(x):
    return 1.0 / (1.0 + 25.0 * x**2)

x_plot = np.linspace(-1.0, 1.0, 1000)
y_true = runge(x_plot)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, n in zip(axes, [9, 17]):
    x_equal = np.linspace(-1.0, 1.0, n)
    y_equal = runge(x_equal)
    y_interp = lagrange_interpolate(x_equal, y_equal, x_plot)
    ax.plot(x_plot, y_true, color="black", label="Runge 函数")
    ax.plot(x_plot, y_interp, label=f"等距节点，节点数={n}")
    ax.scatter(x_equal, y_equal, s=18, zorder=3)
    ax.set_xlabel("x")
    ax.set_title(f"{n - 1} 次插值多项式")
axes[0].set_ylabel("函数值")
axes[0].legend()
fig.suptitle("等距节点上的端点振荡")
fig.tight_layout()
plt.show()


## 切比雪夫节点

切比雪夫节点（Chebyshev nodes）在区间端点附近更密集，可以减小节点乘积的增长。它不能解决所有插值问题，但在光滑函数的全局多项式插值中非常重要。


In [ ]:
node_counts = [9, 17, 25]
for n in node_counts:
    equal_nodes = np.linspace(-1.0, 1.0, n)
    cheb_nodes = chebyshev_nodes(-1.0, 1.0, n)

    equal_error = np.max(np.abs(lagrange_interpolate(equal_nodes, runge(equal_nodes), x_plot) - y_true))
    cheb_error = np.max(np.abs(lagrange_interpolate(cheb_nodes, runge(cheb_nodes), x_plot) - y_true))
    print(f"节点数={n:2d}, 等距节点误差={equal_error:10.3e}, 切比雪夫节点误差={cheb_error:10.3e}")

n = 17
x_equal = np.linspace(-1.0, 1.0, n)
x_cheb = chebyshev_nodes(-1.0, 1.0, n)

plt.figure(figsize=(8, 4.5))
plt.plot(x_plot, y_true, color="black", label="Runge 函数")
plt.plot(x_plot, lagrange_interpolate(x_equal, runge(x_equal), x_plot), label="等距节点")
plt.plot(x_plot, lagrange_interpolate(x_cheb, runge(x_cheb), x_plot), label="切比雪夫节点")
plt.scatter(x_equal, runge(x_equal), s=18, alpha=0.7)
plt.scatter(x_cheb, runge(x_cheb), s=18, alpha=0.7)
plt.xlabel("x")
plt.ylabel("函数值")
plt.title("切比雪夫节点可以减弱端点振荡")
plt.legend()
plt.tight_layout()
plt.show()


## 小结

* 拉格朗日形式直观，适合理解插值基函数。
* 牛顿形式适合增量添加节点和构造差商表。
* 等距节点上的高次全局插值可能严重失效。
* 对有界区间上的光滑函数，切比雪夫节点是重要的稳定化工具。
